**What is Inference :** LLM inference is the process of using a trained language model to produce outputs from input text.

**Training** is nothing but process where model learn it's parameters 
**Inference** is process those learned parameters used to answer a prompt.


At a high level, inference is about one thing: given the tokens seen so far, predict the most likely next token, then repeat.

For example, if the input is:

"The capital of USA is"
the model computes probabilities over the vocabulary and may assign the highest probability to "Washington D,C". Then "Washington,D.C" is appended to the sequence, and the process repeats until a stopping condition is reached.

## 1.The inference pipeline

A typical LLM inference pipeline looks like this:

1. tokenization
2. embedding lookup
3. transformer forward pass
4. logits over vocabulary
5. sampling or decoding
6. next token
7. repeat until stop




**Tokenization**
The model does not directly process raw text. It processes tokens. A token may be a whole word, part of a word, punctuation, or whitespace pattern depending on the tokenizer.

For example:

"unbelievable" might become:
"un", "believ", "able"

or something similar depending on the tokenizer.

The length of the input in tokens matters a lot because computation and memory usage depend heavily on sequence length.



**Embedding**

Each token ID is mapped to a vector of fixed size called the hidden size or model dimension. If the model dimension is 4096, then each token becomes a 4096-dimensional vector.

If your prompt has 1000 tokens, the input tensor before entering the transformer stack is roughly:

1000 × 4096

ignoring the batch dimension for the moment.



**Transformer forward pass**

The embeddings pass through multiple transformer layers. Each layer usually contains:

self-attention
feed-forward network
residual connections
layer normalization

During inference, the model processes the prompt and produces hidden states, then converts the hidden state of the last position into logits over the vocabulary.


**Logits and decoding**

The output of the model at each step is a vector of scores, one per vocabulary token. These scores are called logits.

Then a decoding strategy chooses the next token. Common choices are:

greedy decoding: choose the highest-probability token
sampling: randomly sample according to probabilities
top-k sampling: sample only from the top k tokens
top-p sampling: sample from the smallest set of tokens whose total probability exceeds p
beam search: keep several candidate sequences at once

Once the next token is selected, it is appended to the input and the process repeats.

This repeated generation is why inference is called autoregressive.

## 2. Prefill and decode  

A very important distinction in LLM inference is between the prefill phase and the decode phase.

**Prefill**

Prefill is the first pass over the prompt. If the user provides a prompt of 1000 tokens, the model must process all 1000 tokens to build the internal attention states.

This phase is relatively compute-heavy because the model handles the entire input sequence.


**Decode**

After the prompt is processed, generation begins one token at a time. For each newly generated token, the model computes the next output token.

This phase is iterative and often dominates user-visible response time because tokens are generated sequentially.


**In simple terms**:

prefill = understand the prompt
decode = generate the answer token by token

The performance characteristics of these two phases are different:

prefill is usually limited more by compute
decode is often limited more by memory bandwidth and sequential dependency.







## 3. Why inference is expensive

LLM inference is expensive because modern models are very large and have to do many operations at every generated token.

The main costs come from:

reading billions of parameters from memory
running matrix multiplications across many layers
computing attention over the context
repeating this process for every output token

For a long conversation, the cost grows because the model must attend to more previous tokens unless optimization techniques are used.

That is where batching and KV cache become critical.

## 4. Batching

Batching means processing multiple sequences together in one forward pass.

Instead of running the model separately for one request at a time, the server groups several requests into a batch and processes them simultaneously.

**Why batching helps**

Modern GPUs are designed for parallel math. They are much more efficient when working on larger blocks of data than when handling one tiny input at a time.

Suppose you have four user requests:

Request A: "Write a summary"
Request B: "Translate this paragraph"
Request C: "Explain recursion"
Request D: "Generate code"

Without batching, the GPU runs these four requests separately.

With batching, the server combines them into a single tensor with batch size 4 and processes them together.

This improves hardware utilization and raises throughput




### Batch dimension

If one sequence has shape:

sequence_length × hidden_size

then a batch of B sequences has shape:

B × sequence_length × hidden_size

For example:

8 × 1000 × 4096

This means 8 requests, each 1000 tokens long, each token represented by 4096 numbers.



### Types of batching

There are two common types.

**Static batching**

The server waits for a fixed set of requests, then processes them together.

This is simple but can waste time if requests arrive unevenly.

**Dynamic batching**

The server continuously collects requests arriving within a short time window and groups them into a batch.

This is more efficient in production systems because traffic is not perfectly regular.

**Continuous batching**

Advanced LLM serving systems often use continuous batching, where requests at different generation stages are mixed together.

For example, while one request is in prefill, another may already be in decode. The scheduler tries to keep the GPU busy by merging work across requests.

This is more complex but usually gives much better throughput than naive batching



### Padding and variable lengths

A problem with batching is that requests have different lengths.

**Example:**

Request 1: 50 tokens
Request 2: 300 tokens
Request 3: 1000 tokens

To form a rectangular tensor, shorter sequences are padded to match the longest sequence. This can waste computation.

If the max length in a batch is 1000, then even the 50-token request may occupy space as if it were length 1000.

That is why batch construction matters. Good serving systems try to group similarly sized requests together.

### Benefits of batching

Batching improves:

GPU utilization
overall tokens per second
cost efficiency
number of users served at once

### Drawbacks of batching

Batching can worsen:

latency for individual users
padding waste
scheduler complexity
memory pressure

In practice, serving systems must balance batch size carefully.